In [1]:
import os
import numpy as np
import pandas as pd
import pybliometrics
from pybliometrics.scopus import ScopusSearch, AbstractRetrieval
import os
import requests
from clarivate.wos_starter.client.configuration import Configuration
from clarivate.wos_starter.client.api_client import ApiClient
from clarivate.wos_starter.client.api.documents_api import DocumentsApi
from clarivate.wos_starter.client.rest import ApiException
from tqdm import tqdm
import time
from datetime import datetime
from urllib.parse import quote
import re
pybliometrics.init()

In [2]:
def is_not_none(x):
    return x is not None and str(x).strip() not in {"", "None", "nan"}

def quote_scopus(x):
    x = str(x).replace('"', '\\"').strip()
    return f'"{x}"'

def extract_year(row):
    if is_not_none(row.get("coverDate")):
        return str(row["coverDate"])[:4]
    if is_not_none(row.get("coverDisplayDate")):
        m = re.search(r"\d{4}", str(row["coverDisplayDate"]))
        if m:
            return m.group(0)
    return None

def extract_volume(row):
    """
    '593 LNCE' -> '593'
    None -> None
    """
    vol = row.get("volume")
    if not is_not_none(vol):
        return None
    m = re.search(r"\d+", str(vol))
    return m.group(0) if m else str(vol).strip()

def extract_conference_name(row):
    """
    Example:
    '11th European Lean Educator Conference, ELEC 2025'
    -> tries both full title and acronym/year.
    """
    title = row.get("title")
    if not is_not_none(title):
        return []

    title = str(title).strip()
    names = [title]

    # Capture acronym-style chunks such as ELEC 2025, AAB 2024
    acronyms = re.findall(r"\b[A-Z]{2,10}\s?\d{4}\b", title)
    names.extend(acronyms)

    # Remove leading ordinal if useful:
    # '11th European Lean Educator Conference, ELEC 2025'
    # -> 'European Lean Educator Conference'
    cleaned = re.sub(r"^\d+(st|nd|rd|th)\s+", "", title, flags=re.I)
    cleaned = re.sub(r",\s*[A-Z]{2,10}\s?\d{4}$", "", cleaned).strip()
    if cleaned and cleaned != title:
        names.append(cleaned)

    return list(dict.fromkeys(names))

def build_proceedings_query(row, *, include_doctype=True, strict=True):
    """
    Build a Scopus expansion query for Conference Review / Proceedings Review rows.

    Priority:
    1. source_id
    2. publicationName / issn / eIssn
    3. year
    4. volume / issue / article number if available
    5. conference-specific fields inferred from title:
       CONFNAME(...)
       CONF(...)
    6. exclude the conference-review container itself

    strict=False:
        broad enough to find proceedings papers when volume is missing.

    strict=True:
        uses more AND clauses; higher precision, lower recall.
    """

    must = []
    optional_conf = []

    # Strongest bibliographic container fields
    if is_not_none(row.get("source_id")):
        must.append(f"SOURCE-ID({row['source_id']})")

    # elif is_not_none(row.get("publicationName")):
    #     must.append(f"SRCTITLE({quote_scopus(row['publicationName'])})")

    if is_not_none(row.get("issn")):
        optional_conf.append(f"ISSN({row['issn']})")
    # Year is usually essential
    year = extract_year(row)
    if year:
        must.append(f"PUBYEAR = {year}")

    # Volume / issue / article number, if present
    volume = extract_volume(row)
    if volume:
        must.append(f"VOLUME({volume})")

    # Conference fields: collect all possible CONFNAME variants into ONE OR block
    conf_names = []

    if is_not_none(row.get("title")):
        conf_names.append(str(row["title"]).strip())

    conf_names.extend(extract_conference_name(row))

    # Deduplicate while preserving order
    conf_names = list(dict.fromkeys(conf_names))

    confname_clauses = [
        f"CONFNAME({quote_scopus(name)})"
        for name in conf_names
        if is_not_none(name)
    ]

    if confname_clauses:
        must.append("(" + " OR ".join(confname_clauses) + ")")
   

    return " AND ".join(must)

In [3]:
def _get(obj,key,default=None):
    if obj is None:
        return default
    if isinstance(obj,dict):
        return obj.get(key,default)
    return getattr(obj,key,default)

def _join_names(authors):
    if not authors:
        return None

    out = []
    for a in authors:
        display_name = (
            _get(a, "display_name")
            or _get(a, "full_name")
            or _get(a, "name")
            or _get(a, "researcher_id")
        )
        if display_name:
            out.append(display_name)

    return "; ".join(out) if out else None
def _as_list(x):
    if not x:
        return []
    return x if isinstance(x, list) else [x]
def query_ieee(query, csv_name, max_records=2000, page_size=100):
    """Query IEEE Xplore Metadata API and save results to CSV."""
    timestamp = datetime.now().strftime("%y%m%d_%H%M%S")
    print("Searching IEEE Xplore...")

    api_key = os.environ["IEEE_API_KEY"]
    base_url = "https://ieeexploreapi.ieee.org/api/v1/search/articles"

    rows = []
    start_record = 1

    while len(rows) < max_records:
        params = {
            "apikey": api_key,
            "format": "json",
            "max_records": min(page_size, max_records - len(rows)),
            "start_record": start_record,
            "sort_order": "desc",
            "sort_field": "publication_year",
            "querytext": query,
        }

        resp = requests.get(base_url, params=params, timeout=30)

        if resp.status_code != 200:
            print("Status:", resp.status_code)
            print("Body:", resp.text[:1000])
            break

        data = resp.json()
        articles = data.get("articles", [])

        if not articles:
            break

        for r in tqdm(articles, desc=f"Records {start_record}-{start_record + len(articles) - 1}"):
            authors_data = r.get("authors", {}).get("authors", [])
            authors = "; ".join(
                a.get("full_name") or a.get("name", "")
                for a in authors_data
                if a.get("full_name") or a.get("name")
            ) or None

            keywords_parts = []
            for group in _as_list(r.get("index_terms", {}).get("terms")):
                if isinstance(group, dict):
                    keywords_parts.extend(_as_list(group.get("term")))
                elif isinstance(group, str):
                    keywords_parts.append(group)

            rows.append({
                "source_db": "IEEE Xplore",
                "id": r.get("article_number"),
                "doi": r.get("doi"),
                "item_type":r.get('content_type'),
                "title": r.get("title"),
                "authors": authors,
                "year": r.get("publication_year"),
                "abstract": r.get("abstract"),
                "keywords": "; ".join(str(k) for k in keywords_parts) if keywords_parts else None,
                "citedby_count": r.get("citing_paper_count"),
                "journal_or_venue": (
                    r.get("publication_title")
                    or r.get("conference_title")
                    or r.get("publisher")
                ),
                "affiliation_country":authors_data[0].get('affiliation') if authors_data else None,
                "task": csv_name
            })

        total_records = int(data.get("total_records", len(rows)))

        if start_record + len(articles) > total_records:
            break

        start_record += len(articles)
        time.sleep(0.3)

    df_all = pd.DataFrame(rows)

    print(f"Combined total: {len(df_all)}")

    if not df_all.empty:
        df_all["doi_norm"] = df_all["doi"].astype(str).str.lower().str.strip()
        df_all = df_all.drop_duplicates(subset=["doi_norm", "title"], keep="first")
        df_all.drop(columns=["doi_norm"], inplace=True)

    df_all.to_csv(f'HTDaA_SLR_data/{timestamp}_'+csv_name + ".csv", index=False)

    print(f"✅ Saved IEEE dataset: {csv_name}.csv")

    return df_all

def clean_volume(volume):
    """
    Example:
    '593 LNCE' -> '593'
    """
    if not volume:
        return None
    m = re.search(r"\d+", str(volume))
    return m.group(0) if m else str(volume)

def q(s):
    return '"' + str(s).replace('"', '\\"') + '"'

scop_demolition_terms = [
    "disassembly", "deconstr*", "dismant*", "demol*",
    "refab*", "refurb*", "remanufact*"
]

scop_action_terms = [
    "cut*", "saw*", "jet*", "sort*", "contain*",
    "tool*", "mechan*", "machin*"
]

scop_built_env_terms = [
    "building-construction",
    "building-component",
    "built environment",
    "building material",
    "construction material"
]

scop_robotics_groups = [
    ["robot*", "automat*"],   # robot* AND automat*
    ["autonom*"]             # OR autonom*
]

excl_terms = [
    "chemi*", "bio*", "medical", "protein", "colonialism",
    "history", "identity", "trans", "social", "road",
    "pavement", "mortar", "aggregate", "aggregates"
]


def normalize_text(x):
    if x is None:
        return ""
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9\s\-]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def term_to_regex(term):
    """
    Converts Scopus-style wildcard terms to regex.
    deconstr* -> deconstr[a-z0-9]*
    built environment -> built\s+environment
    building-construction -> building[-\s]?construction
    """
    term = term.lower().strip().strip('"')

    escaped = re.escape(term)
    escaped = escaped.replace(r"\*", r"[a-z0-9]*")
    escaped = escaped.replace(r"\ ", r"\s+")
    escaped = escaped.replace(r"\-", r"[-\s]?")

    return r"\b" + escaped + r"\b"


def has_term(text, term):
    return re.search(term_to_regex(term), text) is not None

def has_any(text, terms):
    return any(has_term(text, t) for t in terms)


def has_all(text, terms):
    return all(has_term(text, t) for t in terms)

def title_abs_key_text(row):
    """
    Recreate TITLE-ABS-KEY from the available Scopus fields.
    """
    fields = [
        row.get("title", ""),
        row.get("abstract", ""),
        row.get("keywords", ""),
    ]

    return normalize_text(" ".join(str(f) for f in fields if f is not None))



def is_relevant(row, csv_name,return_reason=False):
    text = title_abs_key_text(row)
    if row['item_type'] == 'Conference Review': return False, 'is proceeding'
    excluded_hits = [t for t in excl_terms if has_term(text, t)]
    if excluded_hits:
        return (False, {"excluded_hits": excluded_hits}) if return_reason else False
    
    demolition_hit = has_any(text, scop_demolition_terms)
    action_hit = has_any(text, scop_action_terms)
    built_env_hit = has_any(text, scop_built_env_terms)

    robotics_hit = any(
        has_all(text, group)
        for group in scop_robotics_groups
    )

    if csv_name == 'sco_dec_rob_be':
        relevant = (
            demolition_hit
            and action_hit
            and built_env_hit
            and robotics_hit
        )
    
    elif csv_name=='sco_dec_rob':
        relevant = (
            demolition_hit
            and action_hit
            and robotics_hit
        )
    elif csv_name == 'sco_rob_be':
        relevant = (
            built_env_hit
            and robotics_hit
        )
        
    if return_reason:
        return relevant, {
            "demolition_hit": demolition_hit,
            "action_hit": action_hit,
            "built_env_hit": built_env_hit,
            "robotics_hit": robotics_hit,
            "excluded_hits": excluded_hits
        }

    return relevant 
    

def scopus_to_df(results, parent_eid=None, csv_name=None):
    rows = []
    
    for r in tqdm(results):
        r = r._asdict()
        eid = r.get("eid")
        ar = None
        try:
            ar = AbstractRetrieval(eid, view="FULL")
        except Exception:
            pass
        
        rows.append({
                "source_db": "Scopus",
                "id": eid,
                "doi": getattr(ar, "doi", None) if ar else None,
                "item_type":r.get('subtypeDescription'),
                "title": r.get("title"),
                "authors": r['author_names'],
                "year": r.get("coverDate", "")[:4] if r.get("coverDate") else None,
                "abstract": getattr(ar, "abstract", None),
                "keywords": "; ".join(ar.authkeywords) if ar and getattr(ar, "authkeywords", None) else None,
                "citedby_count": getattr(ar, "citedby_count", None),
                "journal_or_venue": r.get("publicationName"),
                "affiliation_country":r.get('affiliation_country'),
                "task": csv_name
                })

    return rows


def query_scopus(query,csv_name):
    """Query Scopus via pybliometrics"""
    timestamp = datetime.now().strftime("%y%m%d_%H%M%S")
    print("Searching Scopus...")
    ss = ScopusSearch(query,verbose=True)
    results = ss.results
    seen_eids = set()
    seen_titles = set()
    unique_results = []

    for r in results:
        if r.eid not in seen_eids and r.title not in seen_titles:
            unique_results.append(r)
            seen_eids.add(r.eid)
            seen_titles.add(r.title)
    dict_results = [r._asdict() for r in unique_results]
    conf_proc = [cr for cr in dict_results if cr['subtype'] in ['pr','cr']]
    print(f"{len(conf_proc)} conference proceedings")
    rows = []
    for r in tqdm(dict_results):
        eid = r.get("eid")
        ar = None
        try:
            ar = AbstractRetrieval(eid, view="FULL")
        except Exception:
            pass
        if r['subtype'] in ['pr','cr']:
            print("expanding query")
            expand_query = build_proceedings_query(r)
            print(expand_query)
            try:
                s = ScopusSearch(expand_query)
                print(len(s.results))
            except Exception as e:
                print(e)
                pass
            count = 0
            if s.results:
                expanded_rows = scopus_to_df(s.results,csv_name=csv_name)
                for e_row in expanded_rows: 
                    keep, reason = is_relevant(e_row,csv_name,return_reason=True)
                    if keep: 
                        rows.append(e_row)
                        count+=1
            print(f"added {count} from expanding search")
                        
                
        if r['subtype'] not in ['pr','cr']:
            rows.append({
                "source_db": "Scopus",
                "id": eid,
                "doi": getattr(ar, "doi", None) if ar else None,
                "item_type":r.get('subtypeDescription'),
                "title": r.get("title"),
                "authors": r['author_names'],
                "year": r.get("coverDate", "")[:4] if r.get("coverDate") else None,
                "abstract": getattr(ar, "abstract", None),
                "keywords": "; ".join(ar.authkeywords) if ar and getattr(ar, "authkeywords", None) else None,
                "citedby_count": getattr(ar, "citedby_count", None),
                "journal_or_venue": r.get("publicationName"),
                "affiliation_country":r.get('affiliation_country'),
                "task": csv_name
                })
            time.sleep(0.3)
    df_all = pd.DataFrame(rows)
    print(f"Combined total: {len(df_all)}")

    df_all['doi_norm'] = df_all['doi'].str.lower().str.strip()
    df_all = df_all.drop_duplicates(subset=['doi_norm', 'title'], keep='first')
    df_all.drop(columns=['doi_norm'], inplace=True)

    df_all.to_csv(f'HTDaA_SLR_data/{timestamp}_'+csv_name+'.csv', index=False)
    print(f"✅ Saved merged dataset: {csv_name}")  
    return df_all  
def query_wos(query, csv_name, db="WOS", limit=50, max_pages=None):
    """Query Web of Science Starter API and save results to CSV."""
    
    timestamp = datetime.now().strftime("%y%m%d_%H%M%S")
    print("Searching Web of Science...")

    configuration = Configuration(
        host="https://api.clarivate.com/apis/wos-starter/v1"
    )
    
    
    configuration.api_key["ClarivateApiKeyAuth"] = os.environ["WOS_API_KEY"]

    rows = []

    with ApiClient(configuration) as api_client:
        api_instance = DocumentsApi(api_client)

        page = 1

        while True:
            try:
                api_response = api_instance.documents_get(
                    q=query,
                    db=db,
                    limit=limit,
                    page=page,
                    sort_field="LD+D"
                )
            except ApiException as e:
                print("Status:", e.status)
                print("Reason:", e.reason)
                print("Body:", e.body)
                break

            hits = getattr(api_response, "hits", None)

            if not hits:
                break

            for r in tqdm(hits, desc=f"Page {page}"):
                uid = _get(r, "uid")
                title = _get(r, "title")
                doi = r.identifiers.doi
                pub_year = (
                    _get(r, "publish_year")
                    or _get(r, "publication_year")
                    or _get(r, "year")
                )

                authors = (
                    _get(r, "names", {}).get("authors")
                    if isinstance(_get(r, "names"), dict)
                    else _get(_get(r, "names"), "authors")
                )

                keywords = _get(r, "keywords")
                if isinstance(keywords, list):
                    keywords = "; ".join(str(k) for k in keywords)

                rows.append({
                    "source_db": "Web of Science",
                    "id": uid,
                    "doi": doi,
                    "item_type":r.source_types[0],
                    "title": title,
                    "authors": _join_names(authors),
                    "year": r.source.publish_year,
                    "abstract": _get(r, "abstract"),
                    "keywords": keywords,
                    "citedby_count": _get(r, "times_cited"),
                    "journal_or_venue": r.source.source_title,
                    "affiliation_country":r.to_dict().get('affiliation_country'),
                    "task": csv_name
                })

            if len(hits) < limit:
                break

            page += 1

            if max_pages and page > max_pages:
                break

            time.sleep(0.3)

    df_all = pd.DataFrame(rows)

    print(f"Combined total: {len(df_all)}")

    if not df_all.empty:
        df_all["doi_norm"] = df_all["doi"].astype(str).str.lower().str.strip()
        df_all = df_all.drop_duplicates(subset=["doi_norm", "title"], keep="first")
        df_all.drop(columns=["doi_norm"], inplace=True)

    df_all.to_csv(f'HTDaA_SLR_data/{timestamp}_'+csv_name + ".csv", index=False)

    print(f"✅ Saved merged dataset: {csv_name}.csv")

    return df_all

In [4]:
def quick_query(query):
    api_key ='288fa70105ec49715973afb65623a485'
    base_url = "https://api.elsevier.com/content/search/scopus"
    headers = {"X-ELS-APIKey": api_key}
    params = {"query": query, "count": 1}
    try:
        response = requests.get(base_url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        total = int(data["search-results"]["opensearch:totalResults"])
        return total
    except Exception as e:
        print(f"Error: {e}")
        return None

In [8]:
scop_demolition_kw = '(disassembly OR deconstr* OR dismant* OR demol* OR refab* OR refurb* OR remanufact*)'
scop_robotics_kw = '(("robot*" AND "automat*") OR "autonom*")'
scop_built_env_kw = '("building-construction" OR "building-component" OR "built environment" OR "building material" OR "construction material" )'
excl_kw = 'chemi* OR bio* OR medical OR protein OR colonialism OR history OR identity OR trans OR social OR road OR pavement OR mortar OR aggregate'

In [9]:
scop_dec_rob_be = f'''TITLE-ABS-KEY({scop_demolition_kw} AND {scop_built_env_kw} AND {scop_robotics_kw} AND NOT ({excl_kw}))'''
scop_dec_rob = f'''TITLE-ABS-KEY({scop_demolition_kw} AND {scop_robotics_kw} AND NOT ({excl_kw}))'''
scop_rob_be = f'''TITLE-ABS-KEY({scop_built_env_kw} AND {scop_robotics_kw} AND NOT ({excl_kw}))'''
print(scop_dec_rob_be)
print(scop_dec_rob)
print(scop_rob_be)

TITLE-ABS-KEY((disassembly OR deconstr* OR dismant* OR demol* OR refab* OR refurb* OR remanufact*) AND ("building-construction" OR "building-component" OR "built environment" OR "building material" OR "construction material" ) AND (("robot*" AND "automat*") OR "autonom*") AND NOT (chemi* OR bio* OR medical OR protein OR colonialism OR history OR identity OR trans OR social OR road OR pavement OR mortar OR aggregate))
TITLE-ABS-KEY((disassembly OR deconstr* OR dismant* OR demol* OR refab* OR refurb* OR remanufact*) AND (("robot*" AND "automat*") OR "autonom*") AND NOT (chemi* OR bio* OR medical OR protein OR colonialism OR history OR identity OR trans OR social OR road OR pavement OR mortar OR aggregate))
TITLE-ABS-KEY(("building-construction" OR "building-component" OR "built environment" OR "building material" OR "construction material" ) AND (("robot*" AND "automat*") OR "autonom*") AND NOT (chemi* OR bio* OR medical OR protein OR colonialism OR history OR identity OR trans OR social

In [10]:
print(quick_query(scop_dec_rob_be))
print(quick_query(scop_dec_rob))
print(quick_query(scop_rob_be))

41
1808
1203


In [156]:
scop_dec_rob_be_df = query_scopus(scop_dec_rob_be,'sco_dec_rob_be')


Searching Scopus...


100%|██████████| 2/2 [00:00<00:00,  1.25it/s]


2 conference proceedings


 29%|██▉       | 12/41 [00:03<00:09,  3.22it/s]

expanding query
SOURCE-ID(21100889404) AND PUBYEAR = 2025 AND VOLUME(593) AND (CONFNAME("2nd International Conference on Architecture Across Boundaries, AAB 2024") OR CONFNAME("AAB 2024") OR CONFNAME("International Conference on Architecture Across Boundaries"))
54


...

expanding query
SOURCE-ID(21100889404) AND PUBYEAR = 2024 AND VOLUME(451) AND (CONFNAME("International Conference on Sustainable Built Environment, ICSBE 2023") OR CONFNAME("ICSBE 2023") OR CONFNAME("International Conference on Sustainable Built Environment"))
27


100%|██████████| 27/27 [00:00<00:00, 853.46it/s]


added 0 from expanding search


100%|██████████| 41/41 [00:12<00:00,  3.32it/s]

Combined total: 39
✅ Saved merged dataset: sco_dec_rob_be


In [157]:
scop_dec_rob_df = query_scopus(scop_dec_rob,'sco_dec_rob')

Searching Scopus...


100%|██████████| 72/72 [01:08<00:00,  1.04it/s]


56 conference proceedings


  6%|▋         | 112/1784 [00:57<15:33,  1.79it/s]

expanding query
SOURCE-ID(21100431311) AND PUBYEAR = 2026 AND (CONFNAME("11th European Lean Educator Conference, ELEC 2025") OR CONFNAME("ELEC 2025") OR CONFNAME("European Lean Educator Conference"))
64


...

expanding query
SOURCE-ID(25674) AND PUBYEAR = 1996 AND VOLUME(1108) AND (CONFNAME("3rd International Conference on Computer Aided Learning and Instruction in Science and Engineering, CALISCE 1996") OR CONFNAME("CALISCE 1996") OR CONFNAME("International Conference on Computer Aided Learning and Instruction in Science and Engineering"))
68


 93%|█████████▎| 1666/1784 [14:34<00:45,  2.58it/s]

added 0 from expanding search


100%|██████████| 1784/1784 [15:20<00:00,  1.94it/s]

Combined total: 1739
✅ Saved merged dataset: sco_dec_rob


In [158]:
scop_rob_be_df = query_scopus(scop_rob_be,'sco_rob_be')

Searching Scopus...


100%|██████████| 48/48 [00:45<00:00,  1.03it/s]


32 conference proceedings


  4%|▍         | 46/1187 [00:16<08:13,  2.31it/s]

expanding query
SOURCE-ID(17700155007) AND PUBYEAR = 2026 AND VOLUME(2790) AND (CONFNAME("5th International Conference on Advanced Research in Technologies, Information, Innovation and Sustainability 2025, ARTIIS 2025") OR CONFNAME("ARTIIS 2025") OR CONFNAME("International Conference on Advanced Research in Technologies, Information, Innovation and Sustainability 2025"))
30


...

added 0 from expanding search


 97%|█████████▋| 1147/1187 [06:11<00:13,  3.08it/s]

expanding query
SOURCE-ID(25674) AND PUBYEAR = 2001 AND VOLUME(2112) AND (CONFNAME("Lecture Notes in Artificial Intelligence (Subseries of Lecture Notes in Computer Science)"))
object of type 'NoneType' has no len()
added 0 from expanding search


100%|██████████| 1187/1187 [06:23<00:00,  3.09it/s]

Combined total: 1167
✅ Saved merged dataset: sco_rob_be


In [146]:
wos_demolition_kw = '(disassembly OR deconstr* OR dismant* OR demol* OR refab* OR refurb* OR remanufact*)'# AND ("cut*" OR "saw*" OR "jet*" OR "sort*" OR "contain*" OR "tool*" OR "mechan*" OR "machin*")'
wos_built_env_kw ='''("building construction" OR
     "building component*" OR
     "built environment" OR
     "building material*" OR
     "construction material*") '''
wos_robotics_kw ='((robot* AND automat*) OR autonom*)'


In [147]:
wos_dec_rob_be = f'''
TS=(
(
    {wos_demolition_kw}

    AND

    {wos_built_env_kw}

    AND

    {wos_robotics_kw}
)
NOT
(
    {excl_kw}
)
)'''

wos_dec_rob = f'''
TS=(
(
    {wos_demolition_kw}

    AND


    {wos_robotics_kw}
)
NOT
(
    {excl_kw}
)
)'''

wos_rob_be = f'''
TS=(
(

    {wos_built_env_kw}

    AND

    {wos_robotics_kw}
)
NOT
(
    {excl_kw}
)
)'''

In [148]:
wos_rob_be_df = query_wos(wos_rob_be,
               csv_name='wos_rob_be',
               )

Searching Web of Science...


Page 12: 100%|██████████| 9/9 [00:00<00:00, 9007.10it/s]

Combined total: 559
✅ Saved merged dataset: wos_rob_be.csv


In [149]:
wos_dec_rob_df = query_wos(wos_dec_rob,
               csv_name='wos_dec_rob',
               )

Searching Web of Science...


Page 24: 100%|██████████| 24/24 [00:00<00:00, 8003.76it/s]

Combined total: 1174
✅ Saved merged dataset: wos_dec_rob.csv


In [160]:
wos_dec_rob_be_df = query_wos(wos_dec_rob_be,
               csv_name='wos_dec_rob_be',
               )

Searching Web of Science...


Page 1: 100%|██████████| 17/17 [5:33:55<00:00, 1178.55s/it]   


Combined total: 17
✅ Saved merged dataset: wos_dec_rob_be.csv


In [151]:
ieee_demolition_kw = '''(disassembly OR deconstruction OR dismantling OR demolition OR
     refabrication OR refurbishment OR remanufacturing)'''# AND (cutting OR sawing OR jet OR sorting OR containment OR
     # tools OR mechanical OR machining)'''
ieee_built_env_kw = '''("building construction" OR
     "building components" OR
     "built environment" OR
     "building materials" OR
     "construction materials")'''
ieee_robotics_kw ='((robotic AND automation) OR autonomous)'

ieee_excl_kw =''' (
    chemistry OR biology OR medical OR protein OR colonialism OR
    history OR identity OR transgender OR social OR road OR
    pavement OR mortar OR aggregates
)'''

In [152]:
iee_dec_rob_be = f'''
(
    {ieee_demolition_kw}

    AND

    {ieee_built_env_kw}

    AND
    
    {ieee_robotics_kw}
    
)

NOT {ieee_excl_kw}

'''

iee_dec_rob = f'''
(
    {ieee_demolition_kw}


    AND
    
    {ieee_robotics_kw}
    
)

NOT {ieee_excl_kw}

'''

iee_rob_be = f'''
(

    {ieee_built_env_kw}

    AND
    
    {ieee_robotics_kw}
    
)

NOT {ieee_excl_kw}

'''

In [153]:
df_ieee = query_ieee(
    query=iee_dec_rob_be,
    csv_name="iee_dec_rob_be"
)

Searching IEEE Xplore...


Records 1-1: 100%|██████████| 1/1 [00:00<?, ?it/s]

Combined total: 1
✅ Saved IEEE dataset: iee_dec_rob_be.csv


In [154]:
df_ieee = query_ieee(
    query=iee_dec_rob,
    csv_name="iee_dec_rob"
)

Searching IEEE Xplore...


Records 301-318: 100%|██████████| 18/18 [00:00<00:00, 17915.87it/s]

Combined total: 318
✅ Saved IEEE dataset: iee_dec_rob.csv


In [155]:
df_ieee = query_ieee(
    query=iee_rob_be,
    csv_name="iee_rob_be"
)

Searching IEEE Xplore...


Records 379-457: 100%|██████████| 79/79 [00:00<00:00, 26347.81it/s]

Combined total: 457
✅ Saved IEEE dataset: iee_rob_be.csv


In [159]:
topic_sets = {
    'dec_rob_be':'Robotic Deconstruction of the Built Environment',
    'rob_be':'Construction Robotics',
    'dec_rob':'Robotic Disassembly',
    
    
}
topic_dfs = {}
seen_titles = set()
seen_dois = set()

for topic in topic_sets.keys():
    data_folder = 'HTDaA_SLR_data/'
    files = os.listdir(data_folder)
    dfs = []
    print(topic)
    for f in files:
        if os.path.isfile(data_folder+f):
            if topic == f[18:-4]:
                # based on naming convention
                try:
                    
                    dfs.append(pd.read_csv(data_folder+f))
                    print(f"adding {f[14:17]}")
                except:
                    print(f"{f} is empty csv, skip")
                    
    combined_ = pd.concat(dfs,ignore_index=True)
    combined_['doi_norm'] = combined_['doi'].str.lower().str.strip()
    combined_ = combined_.drop_duplicates(subset=['doi_norm', 'title'], keep='first')
    combined_ = combined_.drop_duplicates(subset='title', keep='first',ignore_index=True)
    
    keep_rows = []
    
    for idx,row in combined_.iterrows():
        title = str(row['title']).strip().lower()
        doi = row['doi_norm']
        
        if title in seen_titles or (doi and doi in seen_dois):
            continue

        keep_rows.append(idx)

        seen_titles.add(title)

        if doi:
            seen_dois.add(doi)

        
    combined_ = combined_.loc[keep_rows].reset_index(drop=True)
    combined_.drop(columns=['doi_norm'], inplace=True)
    combined_['Ad_Rob_Imp'] = 0
    combined_['Res_Decon']= 0
    combined_['Direct_App']= 0
    topic_dfs.update({topic:combined_})
    
for topic in topic_dfs.keys():
    print(topic)    
    print(len(topic_dfs[topic]))
    topic_dfs[topic].to_csv('HTDaA_SLR_screening/'+topic+'_for_screening.csv', index=False)



dec_rob_be
adding wos
adding iee
adding sco
rob_be
adding wos
adding iee
adding sco
dec_rob
adding wos
adding iee
adding sco
dec_rob_be
35
rob_be
1332
dec_rob
1769
